# Static Diagnosis — What Can Be Learned From the Model Before Solving (Scale, Block Structure, Symmetry)

Previous notebooks (1 to 3) handled dynamic metrics obtained by "actually solving" the model.
Conversely, `minlpkit.collectors.static_diag` and `minlpkit.collectors.symmetry` diagnose numerical soundness and decomposability **solely from the model's coefficients and incidence structure, without ever running branch-and-bound**.
We look at three types of static diagnoses, each using a model where its characteristics are most prominent.

1. **Coefficient Scale** (`extract_coefficients`/`scale_summary`) — Detection of Big-M candidates.
   `samples/scheduling/unit_commitment.py` (known Big-M ratio of 840)
2. **Block Structure** (`incidence`/`reorder_blocks`/`linking_constraints`) — Decomposability.
   `samples/others/scheduling_plant.py` (known linking constraints across jobs)
3. **Symmetry** (`detect_symmetry`) — Interchangeable groups of variables.
   `samples/others/parallel_machines.py` (identical parallel machines, a symmetric model newly created for verification)

All three serve as primary information for the `numerical_scale`, `decomposable`, and `symmetry_info` diagnostic rules.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others", "samples/scheduling"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import unit_commitment as uc
import scheduling_plant as sp
import parallel_machines as pm
from minlpkit.collectors.static_diag import (
    extract_coefficients, scale_summary, incidence, reorder_blocks, linking_constraints,
    residual_scale)
from minlpkit.collectors.symmetry import detect_symmetry

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
SEQ_BLUE = [[0.0, C["surface"]], [1.0, C["s1"]]]
SRC_COLOR = {"制約係数": C["s1"], "RHS/LHS": C["s2"], "目的係数": "#e87ba4", "変数境界": "#eda100"}
print("repo root:", ROOT)

## 1. Coefficient Scale — Detection of Big-M Candidates

`extract_coefficients` gathers the absolute values of linear constraint coefficients, RHS/LHS, objective coefficients, and variable bounds into a single `DataFrame` along with their sources. `scale_summary` extracts the overall max/min ratio and **constraint coefficients/variable bounds that exceed 100 times the median** as Big-M candidates. In Unit Commitment (`unit_commitment.py`), the ramp constraint `pmax=400` is a known Big-M candidate.

This section treats coefficient scale as a **diagnosis** alongside block structure and symmetry.
To see the measurement of the effect of rescaling as a solution itself (before/after comparison centered on $\kappa(A)$), refer to [improve/08_condition_number.ipynb](../improve/08_condition_number.html).

In [ ]:
df_uc = extract_coefficients(uc.build_model())
s_uc = scale_summary(df_uc)
print(f"係数レンジ |min|={s_uc['min']:.3g} |max|={s_uc['max']:.3g} 比={s_uc['ratio']:.3g}")
print(f"Big-M候補({len(s_uc['bigm'])}件):",
     ", ".join(f"{b['name']}={b['magnitude']:.0f}" for b in s_uc["bigm"][:5]))

fig = go.Figure()
for src, color in SRC_COLOR.items():
    dd = df_uc[df_uc["source"] == src]
    if dd.empty:
        continue
    fig.add_trace(go.Box(y=dd["magnitude"], name=src, marker=dict(color=color), boxpoints="outliers"))
fig.update_layout(
    title=dict(text="unit_commitment: 係数の絶対値レンジ(出所別・対数)— 外れ点=Big-M候補",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), showlegend=False,
    yaxis=dict(title="|値|(対数)", type="log", gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=16, t=44, b=36), height=340)
show(fig)

Judging it as "ill-conditioned" based solely on the pre-presolve ratio can lead to false positives. Because SCIP's presolve may tighten Big-M automatically, the `numerical_scale` rule makes its judgment based on the **residual scale after presolve** (`residual_scale`). The ramp constraint in unit_commitment survives (= is not fully tightened by presolve) and thus actually triggers the rule.

In [ ]:
res = residual_scale(uc.build_model())
print(f"presolve前 比={s_uc['ratio']:.3g}  ->  presolve後(残存) 比={res['ratio']:.3g}")
report = mk.analyze(uc.build_model, name="uc-static", time_limit=10)
fired = [f for f in report.findings if f["id"] == "numerical_scale"]
print("numerical_scale 発火:", bool(fired))
if fired:
    print(" 根拠:", fired[0]["evidence"])

## 2. Block Structure — Constraint-Variable Incidence Matrix and RCM Reordering

`incidence` creates a 0/1 constraint × variable incidence matrix (`getConsVars` works even for nonlinear constraints).
By treating this as a bipartite graph and reordering it with `reorder_blocks` (Reverse Cuthill-McKee), decomposable structures emerge as **diagonal blocks**. `linking_constraints` counts how many "variable groups" (the last token of the name, e.g., `J5`/`M2`) each constraint spans, identifying constraints that span many groups as **linking constraints** that form the boundary of decomposition.

In [ ]:
model_plant = sp.build_model()
M, cn, vn = incidence(model_plant)
rp, cp, Mr = reorder_blocks(M)
fig = go.Figure(go.Heatmap(z=Mr, colorscale=SEQ_BLUE, showscale=False, xgap=0, ygap=0, hoverinfo="skip"))
fig.update_layout(
    title=dict(text=f"scheduling_plant: 制約-変数 接続行列(RCM並べ替え {M.shape[0]}×{M.shape[1]})"
                    " — 対角ブロック=分解可能",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="変数(並べ替え後)", showticklabels=False),
    yaxis=dict(title="制約(並べ替え後)", showticklabels=False, autorange="reversed"),
    margin=dict(l=60, r=16, t=44, b=36), height=420)
show(fig)

In [ ]:
lk = linking_constraints(model_plant).head(12).iloc[::-1]
maxg = lk["n_groups"].max()
colors = ["#d03b3b" if g == maxg else C["s1"] for g in lk["n_groups"]]
fig = go.Figure(go.Bar(x=lk["n_groups"], y=lk["constraint"], orientation="h",
    marker=dict(color=colors), text=lk["n_groups"], textposition="outside",
    customdata=lk[["n_vars", "groups"]],
    hovertemplate="%{y}<br>またがるグループ数 %{x}<br>変数 %{customdata[0]}<extra></extra>"))
fig.update_layout(
    title=dict(text="結合制約 = またがる変数グループ数(赤=最多=分解の境界)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="またがるグループ数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=90, r=48, t=44, b=36), height=340)
show(fig)
print("最多結合制約:", lk.iloc[-1]["constraint"], "->", lk.iloc[-1]["groups"])

`load_M1`/`load_M2` (machine load constraints) are extracted as linking constraints spanning all job groups.
This serves as the basis for deciding that they will form the boundary for Benders/Dantzig-Wolfe decomposition.
The `decomposable` rule fires when "maximum linkage is $\ge 4$ groups AND there are $\le 3$ heavy linking constraints"
(The fewer the linking constraints, the smaller the master problem can be kept).

## 3. Symmetry — Detecting Interchangeable Variable Groups via 1-Hop Color Refinement

`detect_symmetry` creates a signature for each variable from (type, objective coefficient, bounds, and the shape and own coefficient of all constraints it belongs to) and returns groups of variables with identical signatures (size $\ge 2$) as symmetry candidates. It can only make a sound judgment (`sound=True`) when all constraints are linear——because if nonlinear constraints exist, signatures based only on the linear parts are incomplete, and the possibility that symmetry is broken by a constant difference cannot be ruled out.
`parallel_machines.py` is a job assignment to identical machines (4 units), a verification model intentionally possessing strong symmetry (newly created to verify the soundness of this collector).

In [ ]:
sym_df, sym_s = detect_symmetry(pm.build_model())
print(f"対称性: {'あり' if sym_s['has_symmetry'] else 'なし'}  健全判定: {sym_s['sound']}  "
     f"群数: {sym_s['n_groups']}  最大群サイズ: {sym_s['largest_group']}  "
     f"対称変数: {sym_s['n_symmetric_vars']}/{sym_s['n_linear_vars']}")

d = sym_df.copy()
d["label"] = d.apply(lambda r: f"群{r['signature_id']} ({r['size']}変数)", axis=1)
d = d.iloc[::-1]
fig = go.Figure(go.Bar(x=d["size"], y=d["label"], orientation="h",
    marker=dict(color="#4a3aa7"), text=d["size"], textposition="outside",
    customdata=d[["members"]], hovertemplate="%{y}<br>%{customdata[0]}<extra></extra>"))
fig.update_layout(
    title=dict(text="parallel_machines: 対称(入替可能)な変数群のサイズ",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="群内の変数数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"]), dtick=1),
    margin=dict(l=140, r=48, t=44, b=36), height=max(220, 60 + 34 * len(d)))
show(fig)

In [ ]:
# 対照: 非線形制約を持つ plant では健全性フラグが False になる(偽陽性回避の仕組み)
_, sym_plant = detect_symmetry(model_plant)
print(f"plant(非線形あり): sound={sym_plant['sound']}  has_symmetry={sym_plant['has_symmetry']}"
     f"  caveat={sym_plant['caveat']}")

report = mk.analyze(pm.build_model, name="parallel-static", time_limit=10)
fired = [f for f in report.findings if f["id"] == "symmetry_info"]
print("symmetry_info 発火:", bool(fired), " (severity=good=対応不要の情報系)")

`symmetry_info` is an informational rule with severity=`good` (= no action required). Because SCIP's built-in symmetry handling (ON by default) automatically breaks symmetry in typical instances, manual lexicographic ordering constraints are usually unnecessary——here too, the diagnosis stops at "detecting the symptom" and reports honestly whether action is even required.

## Summary

- All three types of static diagnoses can be obtained **solely from the model description without solving even once** (symmetry requires no presolve/solve, coefficient scale is checked immediately after construction, and block structure likewise). This is lighter than dynamic diagnoses (Notebooks 1–3) and has the advantage of being executable immediately after writing the model.
- Coefficient scale points out risks of Big-M/numerical instability, block structure points out decomposability, and symmetry points out the risk of search tree explosion; however, **SCIP's presolve/symmetry handling/reduced cost fixing already automatically handles a wide range of these issues**. Therefore, the diagnosis relies on residual values after presolve (`residual_scale`), and symmetry is kept at `good` (reference information)——the consistent design policy of this repository is to "only escalate to `warning`/`serious` things that are truly worth intervening in."

Related: [Methods Guide 0. Diagnosis Itself](../../playbook/00-diagnose.md) /
[Methods Guide 8. Condition Number and Numerical Soundness](../../playbook/08-condition.md) /
[Methods Guide 5. Benders Decomposition](../../playbook/05-benders.md) /
API [`minlpkit.collectors`](../../api/pipeline.md).